Initial thoughts/findings:
* What should we do about low sample numbers for: Northern Ireland; North East England; Wales

In [ ]:
# imports
import pandas as pd
import plotly.express as px

from dap_job_quality import BUCKET_NAME, PROJECT_DIR
from dap_job_quality.getters.healthcare import *

In [ ]:
healthcare_sample = get_salaries_w_soc_region()

# Sample size

In [ ]:
# Count occurrences of each soc_4_digit_name
soc_counts = healthcare_sample['soc_4_digit_name'].value_counts().reset_index()
soc_counts.columns = ['soc_4_digit_name', 'count']

# Create the bar chart
fig = px.bar(soc_counts, 
             y='soc_4_digit_name', 
             x='count', 
            #  title='Counts of SOC 4-Digit Names',
             labels={'soc_4_digit_name': 'SOC 4-Digit Name', 'count': 'Count'},
             text='count')

# Customize layout
fig.update_layout(width=1000,
                  height=600,
                  xaxis={'categoryorder':'total descending'},
                  xaxis_tickangle=-45)  # Rotate labels for better readability

fig.show()

In [4]:
def two_way_heatmap(df, var1 = 'soc_4_digit_name', var2 = 'itl_1_name'):
    # Create a contingency table (cross-tabulation) of counts
    heatmap_data = df.groupby([var1, var2]).size().reset_index(name='count')

    # Create the heatmap
    fig = px.density_heatmap(
        heatmap_data, 
        x=var2, 
        y=var1, 
        z='count', 
        histfunc='sum',
        # title="SOC 4-Digit Name vs ITL 1 Name Heatmap",
        # labels={'itl_1_name': 'ITL 1 Name', 'soc_4_digit_name': 'SOC 4-Digit Name', 'count': 'Count'},
        color_continuous_scale='Viridis',
        text_auto=True,
    )

    fig.update_layout(
        width=1000,  # Adjust width
        height=800,  # Adjust height
        xaxis_tickangle=-45  # Rotate x-axis labels for readability
    )

    return fig

In [ ]:
two_way_heatmap(healthcare_sample, var1 = 'soc_4_digit_name', var2 = 'itl_1_name')

In [6]:
top_professions = ['Other registered nursing professionals',
                   'Other health professionals n.e.c.',
                   'Registered mental health nurses',
                   'Occupational therapists']

In [ ]:
two_way_heatmap(healthcare_sample[healthcare_sample['soc_4_digit_name'].isin(top_professions)], var1 = 'soc_4_digit_name', var2 = 'itl_1_name')

In [ ]:
two_way_heatmap(healthcare_sample[~healthcare_sample['soc_4_digit_name'].isin(top_professions)], var1 = 'soc_4_digit_name', var2 = 'itl_1_name')

# Pay by 4 digit SOC code

In [ ]:
# salary boxplot
fig = px.box(healthcare_sample, y='soc_4_digit_name', x='hourly_wage')

fig.update_layout(width=1000,
                  height=600,
                  xaxis={'categoryorder':'total descending'},
                  )

fig.show()

# Job quality data

In [11]:
job_qual = pd.read_parquet('s3://open-jobs-lake/job_quality/health_social_care/health_jobs_jq_dimensions.parquet')

In [ ]:
job_qual.head()

In [ ]:
job_qual_processed, dimensions_wide = analysis_utils.process_jq_data(job_qual)
job_qual_processed.head()

In [ ]:
dimensions_wide

In [ ]:
sample_w_salaries_dimensions = pd.merge(healthcare_sample, dimensions_wide, on='id', how='left')
sample_w_salaries_dimensions.head()

In [ ]:
sample_w_salaries_dimensions.columns

In [ ]:
prop_table = sample_w_salaries_dimensions.groupby(['soc_4_digit_name']).agg({'FLEX_HOURS': sum, 'FLEX_LOC': sum,'L&D': sum, 'CAREER': sum,'id': 'size', 'hourly_wage': 'median'}).reset_index()

for dim in ['FLEX_HOURS', 'FLEX_LOC','L&D', 'CAREER']:
    prop_table[f'{dim}_perc'] = (prop_table[dim] / prop_table['id']) * 100

prop_table

In [28]:
prop_table[['soc_4_digit_name', 'id', 'hourly_wage', 'CAREER_perc', 'FLEX_HOURS_perc','FLEX_LOC_perc', 'L&D_perc']].to_csv('outputs/sector_multi_comparison.csv')

## Contract type

In [21]:
contracts_full = get_contracts_full()
contract_df = get_contract_per_job()
contracts_complete = get_contracts_complete()

In [ ]:
contract_df['final_contract_type'].value_counts()

In [ ]:
# contracts_complete = contract_df[contract_df['final_contract_type'] != 'Unknown']
len(contracts_complete) / len(healthcare_sample) # We can get contract type for 20% of the sample

In [26]:
contracts_salaries_soc_region = pd.merge(contracts_complete, healthcare_sample, on='id', how='left')

In [ ]:
# Count occurrences of final_contract_type for each soc_4_digit_name
contract_counts = contracts_salaries_soc_region.groupby(['soc_4_digit_name', 'final_contract_type']).size().reset_index(name='count')

def stacked_bar(counts_df, x, y, colour):
    # Create a stacked bar chart
    fig = px.bar(
        counts_df,
        x=x, 
        y=y,
        color=colour,
        orientation='h',  # Horizontal bars
        # title='Distribution of Final Contract Types by SOC 4-Digit Name',
        # labels={'count': 'Count', 'soc_4_digit_name': 'SOC 4-Digit Name'},
        barmode='stack'  # Stacked bars
    )
    
    fig.update_layout(
        width=1000,  # Adjust width
        height=800,  # Adjust height
        # xaxis_tickangle=-45,  # Rotate x-axis labels for readability)
    )

    return fig

stacked_bar(contract_counts, x='count', y='soc_4_digit_name', colour='final_contract_type')


In [ ]:
contract_counts['percentage'] = contract_counts.groupby('soc_4_digit_name')['count'].transform(lambda x: x / x.sum() * 100)

stacked_bar(contract_counts, x='percentage', y='soc_4_digit_name', colour='final_contract_type')